# Step 1 — Disfluency BIO labeling (IP/RP/RM/RC/FS taxonomy)

No disfluency-tagged dataset exists yet. `data/normalized/intent_dataset_normalized.jsonl`'s `text` field keeps disfluency markers (fillers, repeats, false starts, self-corrections); `text_normalized` only strips punctuation, so neither field is a ready-made BIO label. This notebook builds rule-based BIO labels over `text` using a 5-tag taxonomy:

- `IP` (interregnum/filler) — discourse fillers: `eh`, `anu`, `um`, `hmm`, `ee`, `em`
- `FS` (false start) — hyphen-truncated partial word immediately followed by its completed form (e.g. `na- nasi`)
- `RC` (repeat) — exact adjacent word duplication (e.g. `mau, mau`)
- `RP`/`RM` (reparandum/repair) — self-correction pairs bridged by a filler, restricted to numeral-word swaps (e.g. `dua, eh, tiga`) since that's the only pattern in this corpus where the bridge filler genuinely separates a wrong value from its correction rather than just marking mid-clause hesitation

Encoder target for downstream training: `LazarusNLP/NusaBERT-base` (IndoBERT continued-pretrained on Indonesian + regional languages incl. Javanese — better fit than plain IndoBERT given Ordo's Javanese-accented speech).

In [1]:
import json
from collections import Counter
from pathlib import Path

SRC = Path("../data/normalized/intent_dataset_normalized.jsonl")
OUT_DIR = Path("../data/disfluency")

## Step 2 — Label vocabulary and heuristics

`FILLERS` covers standalone discourse fillers tagged `IP`. `BRIDGES` is the subset of fillers (`eh`, `anu`) that can sit between two commas — but a comma-bridge alone doesn't imply a repair; it's only a genuine `RP`/`RM` pair when the tokens flanking it are both numerals with different values (verified against the corpus: `anu`-bridges are always pure hesitation before cancel/query statements, `eh`-bridges are mostly the same, with only numeral swaps being true corrections). `VOCATIVES` excludes address terms (`kak`, `bang`, `mas`, ...) from repeat-detection so `"bang, bang"`-style direct address never gets mistagged as `RC`.

In [2]:
FILLERS = {"eh", "anu", "um", "hmm", "ee", "em"}
BRIDGES = {"eh", "anu"}
VOCATIVES = {"kak", "bang", "mas", "bu", "pak", "mbak", "lo", "gue", "bro", "sis"}
NUMERALS = {
    "satu", "dua", "tiga", "empat", "lima", "enam", "tujuh", "delapan", "sembilan", "sepuluh",
}


def is_numeral(tok: str) -> bool:
    return tok.lower() in NUMERALS or tok.isdigit()


def tokenize(text: str) -> list[str]:
    return text.split()


def strip_punct(tok: str) -> str:
    return tok.strip(",.;:!?")

## Step 3 — BIO labeler

Single left-to-right pass tags `FS` (hyphen-truncated partial word), `RC` (adjacent exact repeat), and standalone `IP` fillers. `apply_repair_bridges` runs as a second pass restricted to comma-bridged numeral swaps, since that pattern needs lookahead/lookbehind across the bridge token rather than a simple sequential scan.

In [3]:
def label_core(toks: list[str], raw_toks: list[str]) -> list[str]:
    n = len(toks)
    labels = ["O"] * n
    i = 0
    while i < n:
        cur = toks[i].lower()
        cur_prefix = cur.rstrip("-")

        # FS: hyphen-truncated partial word followed by its completed form
        if raw_toks[i].rstrip(",").endswith("-") and i + 1 < n and cur_prefix and toks[i + 1].lower().startswith(cur_prefix):
            labels[i] = "B-FS"
            i += 1
            continue

        # RC: exact adjacent repeat (excluding vocatives/address terms)
        if i + 1 < n and cur == toks[i + 1].lower() and cur not in VOCATIVES and cur != "":
            labels[i] = "B-RC"
            labels[i + 1] = "I-RC"
            i += 2
            continue

        # IP: standalone discourse filler
        if cur in FILLERS and cur not in VOCATIVES:
            labels[i] = "B-IP"
            i += 1
            continue

        i += 1

    return labels


def apply_repair_bridges(toks: list[str], raw_toks: list[str], labels: list[str]) -> list[str]:
    """Detect ", <bridge>," with a numeral immediately before and after, differing in
    value (e.g. "dua, eh, tiga"). Restricting to numeral swaps avoids false positives where
    the bridge filler is just mid-clause hesitation, not an actual value correction."""
    for i, t in enumerate(toks):
        if t.lower() in BRIDGES and labels[i] == "B-IP":
            prev_has_comma = i > 0 and raw_toks[i - 1].endswith(",")
            next_has_comma = raw_toks[i].endswith(",")
            if not (prev_has_comma and next_has_comma):
                continue
            rp_idx, rm_idx = i - 1, i + 1
            if rm_idx >= len(toks):
                continue
            rp_tok, rm_tok = toks[rp_idx], toks[rm_idx]
            if is_numeral(rp_tok) and is_numeral(rm_tok) and rp_tok.lower() != rm_tok.lower():
                if labels[rp_idx] == "O":
                    labels[rp_idx] = "B-RP"
                if labels[rm_idx] == "O":
                    labels[rm_idx] = "B-RM"
    return labels


def full_label(text: str) -> tuple[list[str], list[str]]:
    raw_toks = tokenize(text)
    toks = [strip_punct(t) for t in raw_toks]
    labels = label_core(toks, raw_toks)
    labels = apply_repair_bridges(toks, raw_toks, labels)
    return toks, labels

## Step 4 — Validation

Auto-corrects orphan `I-` tags (continuation tag with no preceding `B-`/`I-` of the same type) to `B-`, and any unrecognized label value to `O`. `label_core`/`apply_repair_bridges` never produce orphans by construction, but this guards against future heuristic changes silently breaking BIO well-formedness.

In [4]:
VALID_LABELS = {"O", "B-IP", "B-FS", "B-RC", "I-RC", "B-RP", "B-RM"}


def validate_bio(toks: list[str], labels: list[str], record_id) -> list[str]:
    fixed = list(labels)
    prev_type = None
    for i, lb in enumerate(fixed):
        if lb not in VALID_LABELS:
            print(f"[{record_id}] unknown label {lb!r} at token {toks[i]!r}, coercing to O")
            fixed[i] = "O"
            lb = "O"
        if lb.startswith("I-"):
            tag_type = lb[2:]
            if prev_type != tag_type:
                print(f"[{record_id}] orphan {lb} at token {toks[i]!r}, coercing to B-{tag_type}")
                fixed[i] = f"B-{tag_type}"
        prev_type = fixed[i][2:] if fixed[i] != "O" else None
    return fixed

## Step 5 — Run labeler over the corpus

In [5]:
rows = [json.loads(line) for line in SRC.open()]

labeled_rows = []
tag_counts = Counter()
for row in rows:
    toks, labels = full_label(row["text"])
    labels = validate_bio(toks, labels, row["id"])
    for lb in labels:
        tag_counts[lb] += 1
    labeled_rows.append({
        "id": row["id"],
        "text": row["text"],
        "intent": row["intent"],
        "tokens": toks,
        "labels": labels,
    })

print(f"{len(labeled_rows)} records labeled")
print("tag counts:", dict(tag_counts))

2060 records labeled
tag counts: {'B-IP': 848, 'O': 19588, 'B-RC': 38, 'I-RC': 38, 'B-RP': 32, 'B-RM': 32, 'B-FS': 33}


## Step 6 — Sanity check on sample sentences

Spot-checks the labeler against representative patterns for each tag, including the two genuine numeral-swap repairs and the (correctly excluded) `anu`-bridge hesitation cases.

In [6]:
samples = [
    "eh, gue mau pesen nasi goreng spesial dua porsi",
    "mas, gue mau, mau pesen ayam goreng satu porsi",
    "eh gue pesen mie goreng spesial dua, eh, tiga porsi pak",
    "na- nasi goreng ikan asin satu ya kak",
    "mbak, anu, mie goreng spesialnya itu dibatalin aja deh",
]

for text in samples:
    toks, labels = full_label(text)
    print(text)
    print([f"{t}/{l}" for t, l in zip(toks, labels) if l != "O"] or "no tags")
    print()

eh, gue mau pesen nasi goreng spesial dua porsi
['eh/B-IP']

mas, gue mau, mau pesen ayam goreng satu porsi
['mau/B-RC', 'mau/I-RC']

eh gue pesen mie goreng spesial dua, eh, tiga porsi pak
['eh/B-IP', 'dua/B-RP', 'eh/B-IP', 'tiga/B-RM']

na- nasi goreng ikan asin satu ya kak
['na-/B-FS']

mbak, anu, mie goreng spesialnya itu dibatalin aja deh
['anu/B-IP']



## Step 7 — Align labels to NusaBERT-base WordPiece tokens

`LazarusNLP/NusaBERT-base` is IndoBERT continued-pretrained on Indonesian plus regional languages (incl. Javanese, Sundanese, Minangkabau, ...) — a better fit than plain `indobenchmark/indobert-base-p1` for Ordo's Javanese-accented restaurant speech. Word-level BIO labels align to WordPiece subwords via `word_ids()`: the first subword of each word keeps its label id, continuation subwords and special tokens get `-100` so `CrossEntropyLoss` ignores them during training.

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "LazarusNLP/NusaBERT-base"
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_list = sorted({lb for r in labeled_rows for lb in r["labels"]})
label2id = {lb: i for i, lb in enumerate(label_list)}
id2label = {i: lb for lb, i in label2id.items()}
print(label2id)

{'B-FS': 0, 'B-IP': 1, 'B-RC': 2, 'B-RM': 3, 'B-RP': 4, 'I-RC': 5, 'O': 6}


In [8]:
def align_labels_with_tokens(tokens: list[str], labels: list[str]) -> dict:
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    word_ids = encoding.word_ids()
    aligned = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned.append(-100)
        elif word_id != prev_word_id:
            aligned.append(label2id[labels[word_id]])
        else:
            aligned.append(-100)
        prev_word_id = word_id
    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": aligned,
    }


aligned_rows = []
mismatches = 0
for r in labeled_rows:
    aligned = align_labels_with_tokens(r["tokens"], r["labels"])
    if len(aligned["input_ids"]) != len(aligned["labels"]):
        mismatches += 1
        continue
    aligned_rows.append({
        "id": r["id"],
        "intent": r["intent"],
        **aligned,
    })

print(f"{len(aligned_rows)} aligned, {mismatches} mismatches")

2060 aligned, 0 mismatches


## Step 8 — Dominant disfluency type per record (for stratified split)

Each record gets one dominant tag for stratification — the rarest non-`O` tag type present, falling back to `fluent` for records with no disfluency tags. Prioritizing the rarest tag (rather than e.g. first-occurring) keeps `FS`/`RP`/`RM` (only 3/2/2 occurrences each) from being randomly orphaned into a single split.

In [9]:
TAG_PRIORITY = ["FS", "RP", "RM", "RC", "IP"]


def dominant_type(labels: list[str]) -> str:
    present = {lb[2:] for lb in labels if lb != "O"}
    for tag in TAG_PRIORITY:
        if tag in present:
            return tag
    return "fluent"


dominant_by_id = {r["id"]: dominant_type(r["labels"]) for r in labeled_rows}
print(Counter(dominant_by_id.values()))

Counter({'fluent': 1180, 'IP': 777, 'RC': 38, 'FS': 33, 'RP': 32})


## Step 9 — Stratified 80/10/10 split

`FS`, `RP`, `RM` have only 2-3 records each. `train_test_split` needs >=2 members per class per split, and this runs as *two* sequential splits (80/20, then the 20% into 10/10) — a class needs >=4 total to survive both. Records whose dominant type has fewer than 4 occurrences go entirely to train; the remaining (`fluent`, `IP`, `RC`) stratify normally.

In [10]:
from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRAC = 0.1
TEST_FRAC = 0.1
MIN_STRATIFY_COUNT = 4

counts = Counter(dominant_by_id.values())
forced_train_ids = {rid for rid, dt in dominant_by_id.items() if counts[dt] < MIN_STRATIFY_COUNT}
stratifiable = [r for r in aligned_rows if r["id"] not in forced_train_ids]
forced_train = [r for r in aligned_rows if r["id"] in forced_train_ids]

labels_strat = [dominant_by_id[r["id"]] for r in stratifiable]
train, rest = train_test_split(
    stratifiable, test_size=VAL_FRAC + TEST_FRAC, stratify=labels_strat, random_state=SEED
)
rest_labels = [dominant_by_id[r["id"]] for r in rest]
val, test = train_test_split(
    rest, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=rest_labels, random_state=SEED
)
train = train + forced_train

print(f"train={len(train)} val={len(val)} test={len(test)}")

train=1648 val=206 test=206


In [11]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


OUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(OUT_DIR / "train.jsonl", train)
write_jsonl(OUT_DIR / "val.jsonl", val)
write_jsonl(OUT_DIR / "test.jsonl", test)
json.dump(label2id, (OUT_DIR / "label_map.json").open("w"), indent=2)

print("wrote", OUT_DIR)

wrote ../data/disfluency
